In [47]:
def add_prefix_token(text):
    text = text.replace("\t", " ")
    text = text.strip()
    if text[0].isalpha() or text[3].isalpha():
        return "[SQL]\n" + text
    else:
        return "[LOG]\n" + text


In [43]:
"  cat   ".strip()

'cat'

In [48]:
print(add_prefix_token(
    "1' GROUP BY users.password, users.username HAVING 1=1--"
))


[SQL]
1' GROUP BY users.password, users.username HAVING 1=1--


In [49]:
print(add_prefix_token(
    "2025-01-06 14:33:02 | User: webapp | IP: 88.214.36.89 | SELECT * FROM users"
))


[LOG]
2025-01-06 14:33:02 | User: webapp | IP: 88.214.36.89 | SELECT * FROM users


In [50]:
print(add_prefix_token(
    "2025-12-28T16:57:2.121319Z	91	Query	select EbOSA9IELrHrvJN from 7MrkiJqCjWdPr-8Wm7 where L9yjYu = 'JG_Iz6BNCBXRtzzoL' or 1=1"
))


[LOG]
2025-12-28T16:57:2.121319Z 91 Query select EbOSA9IELrHrvJN from 7MrkiJqCjWdPr-8Wm7 where L9yjYu = 'JG_Iz6BNCBXRtzzoL' or 1=1


In [11]:
def add_prefix_token(raw_text: str) -> str:
    # เติม token เพื่อแยกว่าเป็น extracted sql หรือ full log
    if not raw_text:
        return "[LOG]\n"

    text = raw_text.strip()
    
    # ใช้ regex pattern เพื่อตรวจสอบ SQL keywords ที่ครอบคลุมมากขึ้น
    import re
    
    # รายการ SQL keywords ที่ครอบคลุมมากขึ้น
    sql_keywords = (
        "SELECT", "INSERT", "UPDATE", "DELETE", "WITH", "CREATE", "DROP", 
        "ALTER", "TRUNCATE", "MERGE", "EXPLAIN", "DESCRIBE", "SHOW", "USE",
        "BEGIN", "COMMIT", "ROLLBACK", "SAVEPOINT", "CALL", "DECLARE",
        "EXECUTE", "PREPARE", "GRANT", "REVOKE"
    )
    
    # Pattern สำหรับ SQL statements (รวมถึงคำสั่งที่อาจมี comment นำหน้า)
    sql_patterns = [
        # เริ่มต้นด้วย SQL keyword (อาจมี whitespace หรือ comment นำหน้า)
        r'^\s*(--.*\n\s*)*\s*({})\s+'.format('|'.join(sql_keywords)),
        
        # SQL comment style
        r'^\s*/\*.*\*/\s*({})\s+'.format('|'.join(sql_keywords)),
        
        # CTE (Common Table Expressions) ที่เริ่มด้วย WITH
        r'^\s*WITH\s+',
        
        # Transaction commands
        r'^\s*(BEGIN|START\s+TRANSACTION|COMMIT|ROLLBACK)\s',
    ]
    
    # Pattern สำหรับ log indicators (เพิ่มเติมจากของคุณ)
    log_indicators = [
        r'\|',                     # pipe delimiter
        r'IP:',                    # IP address
        r'timestamp',              # timestamp
        r'\d{4}-\d{2}-\d{2}',     # date pattern
        r'\d{2}:\d{2}:\d{2}',     # time pattern
        r'\[.*?\]',               # bracket patterns (common in logs)
        r'ERROR|WARN|INFO|DEBUG', # log levels
        r'host:|user:|db:',       # common log fields
    ]
    
    # ตรวจสอบว่าเป็น SQL หรือไม่
    is_sql = False
    for pattern in sql_patterns:
        if re.search(pattern, text, re.IGNORECASE | re.DOTALL):
            is_sql = True
            break
    
    # ตรวจสอบว่าเป็น log หรือไม่ (ถ้าไม่ใช่ SQL อย่างชัดเจน)
    is_log = False
    if not is_sql:
        for pattern in log_indicators:
            if re.search(pattern, text, re.IGNORECASE):
                is_log = True
                break
    
    # Heuristic เพิ่มเติม: ตรวจสอบโครงสร้าง SQL
    if not is_log and not is_sql:
        # ถ้าข้อความมี SQL-like patterns
        sql_like_patterns = [
            r'\bFROM\s+\w+',      # มี FROM clause
            r'\bWHERE\s+',        # มี WHERE clause
            r'\bSET\s+\w+\s*=',   # มี SET ใน UPDATE
            r'\bVALUES\s*\(',     # มี VALUES ใน INSERT
            r'\bJOIN\s+\w+',      # มี JOIN
            r'\bGROUP\s+BY\b',    # มี GROUP BY
            r'\bORDER\s+BY\b',    # มี ORDER BY
            r'\bHAVING\s+',       # มี HAVING
        ]
        
        sql_like_count = 0
        for pattern in sql_like_patterns:
            if re.search(pattern, text, re.IGNORECASE):
                sql_like_count += 1
        
        # ถ้ามี pattern คล้าย SQL มากกว่า 1 จุด
        if sql_like_count >= 2:
            is_sql = True
    
    # ตัดสินใจ prefix
    if is_sql:
        return "[SQL]\n" + text
    else:
        # อาจเป็น log หรือ plain text อื่นๆ
        return "[LOG]\n" + text


# Alternative: แบบง่ายกว่าแต่ครอบคลุม keyword มากขึ้น
def add_prefix_token_simple(raw_text: str) -> str:
    if not raw_text:
        return "[LOG]\n"

    text = raw_text.strip()
    
    # SQL keywords ที่ครอบคลุมมากขึ้น (รวม DDL, DML, DCL, TCL)
    sql_keywords = (
        # DML (Data Manipulation Language)
        "SELECT", "INSERT", "UPDATE", "DELETE", "MERGE",
        
        # DDL (Data Definition Language)
        "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "COMMENT",
        
        # DCL (Data Control Language)
        "GRANT", "REVOKE",
        
        # TCL (Transaction Control Language)
        "BEGIN", "COMMIT", "ROLLBACK", "SAVEPOINT", "SET TRANSACTION",
        
        # Query clauses and common commands
        "WITH", "EXPLAIN", "DESC", "DESCRIBE", "SHOW",
        
        # Other SQL commands
        "EXECUTE", "EXEC", "CALL", "DECLARE", "PREPARE",
        
        # PostgreSQL specific
        "VACUUM", "ANALYZE",
        
        # SQL Server specific
        "EXECUTE", "EXEC",
    )
    
    # ตรวจสอบว่าเริ่มต้นด้วย SQL keyword (อาจมี comment หรือ whitespace นำหน้า)
    import re
    
    # ลบ comment และ whitespace นำหน้าออกก่อนตรวจสอบ
    cleaned_text = re.sub(r'^\s*(--.*?\n\s*)*\s*', '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r'^\s*/\*.*?\*/\s*', '', cleaned_text, flags=re.IGNORECASE | re.DOTALL)
    
    # ตรวจสอบว่า text ที่ clean แล้วเริ่มต้นด้วย SQL keyword หรือไม่
    starts_with_sql = cleaned_text.upper().startswith(sql_keywords)
    
    # เงื่อนไขเพิ่มเติมเพื่อแยกแยะ log
    has_log_indicators = (
        "|" in text or
        "IP:" in text or
        "timestamp" in text.lower() or
        re.search(r'\d{4}-\d{2}-\d{2}[T\s]\d{2}:\d{2}:\d{2}', text) or  # ISO timestamp
        re.search(r'\[.*?\]', text) or  # bracket patterns
        any(level in text.upper() for level in ["ERROR", "WARNING", "WARN", "INFO", "DEBUG", "FATAL"])
    )
    
    if starts_with_sql and not has_log_indicators:
        return "[SQL]\n" + text
    else:
        return "[LOG]\n" + text

In [19]:
print(add_prefix_token(
    "EXEC sp_executeSQL N'SELECT * FROM Employees WHERE Name = ''' + @name + ''''"
))


[LOG]
EXEC sp_executeSQL N'SELECT * FROM Employees WHERE Name = ''' + @name + ''''
